# Cuaderno 10 — Fallas, decisiones y por qué

**Para qué sirve este cuaderno.** Un trabajo de fin de máster no se juzga solo por el resultado
final, sino por saber qué se probó, qué falló y cómo se detectó. Aquí queda cada error que
cometimos, cómo apareció, qué se decidió y con qué cifra. No es un anexo de disculpas: es la
evidencia de que el resultado final sobrevivió a una auditoría.

Está organizado por el tipo de problema, que son los cinco que nos señaló el profesor:
**errores de IA, fuga de información, métricas incorrectas, mal manejo de nulos y sobreajuste.**

Cada sección tiene una celda que **reproduce la falla** con los datos reales, para que cualquiera
pueda verla por sí mismo. Las demostraciones son todas rápidas: ninguna toma más de un minuto.

> Convención: cada falla lleva un identificador `F01`, `F02`… que se puede citar desde la tesis.


## 0. Registro de decisiones

| # | Qué se intentó | Qué pasó | Cómo se detectó | Qué se decidió | Cifra |
|---|---|---|---|---|---|
| F01 | Filtrar el RUC del proveedor con `IS NOT NULL` | 42.294 filas con RUC en cadena vacía pasaron el filtro y quedaron agrupadas como un único proveedor | Al revisar el proveedor con más monto de la red | Filtrar con `TRIM(COALESCE(x,'')) <> ''` | 46.370 registros, **4.478,8 MM** (la cifra anterior, 4.335,4, salía de `monto_tender`: ver F14) |
| F02 | Construir los pares del mismo representante a partir de pares que ya competían | El resultado era circular: medía lo que la propia construcción imponía | Al notar que el archivo tenía solo 35 pares y una columna `dominancia_max` | Reconstruir desde el directorio completo de la Superintendencia | 35 pares → 446 procesos, 83 representantes |
| F03 | Excluir variables por AUC alta (umbral 0,80) | Habría eliminado `ratio_monto_capital`, que es legítima, y conservado otras que no | Al revisar una por una el origen de las variables señaladas | La exclusión se decide por **origen**, no por desempeño. La AUC solo avisa | `ratio_monto_capital` AUC 0,820 se conserva; disolución AUC 0,967 se excluye |
| F04 | Calcular el AUC del objetivo B sobre todo el panel | Inflaba las variables que marcan **pertenencia al universo**, porque incluía 215.788 filas que nunca pudieron ser positivas | Al ver variables societarias con AUC alta sin explicación plausible | Calcular dentro del universo en riesgo | `n_parties` 0,800 → 0,734; `sup_anonima` 0,625 → 0,611 |
| F05 | Codificar en 0 los indicadores societarios de las empresas ausentes del directorio | Confundía «no es sociedad anónima» con «no está registrada como compañía»: fuga por definición | Al auditar el manejo de nulos | Dejarlos como ausentes, nunca en 0 | — |
| F06 | Leer el catastro del RUC declarando latin-1 | Además de corromper el texto, el lector descartó en silencio el 79 % de las filas | Al ver cero nombres con tilde o con ñ en 1,7 millones de registros | Declarar utf-8 y verificar el conteo contra el origen | 1.759.893 → 8.458.107 filas |
| F07 | Comparar el modelo contra un nulo centrado en 0,50 | El control negativo parecía fallar: 3 de 5 placebos salían «significativos» | Al correr el control negativo con etiquetas barajadas | Nulo por permutación de etiquetas, repitiendo todo el procedimiento | El nulo real está en 0,486, no en 0,50 |
| F08 | Usar `monto_tender` como variable de monto | Está nulo en las 216.469 subastas inversas y presente en el 100 % de las otras clases: su patrón de nulos **es** el tipo de procedimiento | Al ver que la suma de monto de Subasta Inversa salía nula | Monto canónico `monto_final`; `monto_tender` fuera del modelo | Quitarlo **mejoró** el resultado restringido: 0,760 → 0,797 |
| F09 | Usar el generador de azar global dentro de la evaluación | El resultado dependía de cuántas celdas se corrieron antes: los casos «anticipada» se movieron 0,25 entre dos corridas idénticas | Al comparar dos ejecuciones con el mismo conjunto de variables | Semilla explícita y promedio sobre 5 semillas, con el recorrido declarado | Recorrido 0,034 en A completo; 0,119 en A anticipada |
| F10 | Reportar la tasa de oferente único sobre toda la base | El catálogo electrónico, que es el 83,5 % de los procesos, no tiene proceso de ofertas: la cifra global no describía nada (ver F13) | Al normalizar los 1.644 valores de `tipo_contratacion` | Universo de competencia explícito | 22,6 % en el universo competitivo |
| F11 | Tratar el 83,5 % de nulos de `n_oferentes` como problema de calidad | No faltaban: el catálogo electrónico no tiene oferentes. Eran el 99,999 % de los nulos (los otros 19 están fuera del catálogo) | Al cruzar los nulos con la taxonomía de procedimiento | Declararlo como propiedad del procedimiento, no como límite del dato | 2.285.455 de 2.285.474 nulos |
| F12 | Usar el registro de proveedores incumplidos como etiqueta | 0 aciertos en 800 casos revisados a mano entre tres personas | Revisión manual | Se reporta como resultado y se abandona la etiqueta | Cota superior 95 %: 0,374 % |
| F13 | Calcular la tasa de oferente único sobre toda la base con SQL | `proveedor_unico` está **nulo** en los 2.285.455 procesos de catálogo. El `CASE WHEN … ELSE 0` los contó como «no», y eso produjo un 7,7 % que parecía una tasa y no lo era | Al recalcular lo mismo en pandas, que omite los nulos, y obtener `NaN` donde el SQL había dado 0,0 | La tasa solo se reporta donde la variable existe | 7,7 % (inventado) → 47,0 % donde existe → 22,6 % en el universo competitivo |
| F14 | Publicar la tabla de montos por clase de procedimiento sumando `monto_tender` | `monto_tender` está nulo en las 216.469 subastas inversas, así que el total del «universo competitivo» salió de los otros 39.006 procesos: 22.974,4 MM en lugar de 38.542,2 MM. El total general daba 51.327,3 MM en lugar de 66.570,9 MM | Al cuadrar la tabla del documento contra la salida de F10 y F11, que usa `monto_final`: el total declarado coincidía exactamente con el total de `monto_tender` | Todo peso que se reporte sale de `monto_final`, y la columna de monto se recalcula en las dos tablas de la sección 6 | Al universo competitivo le faltaban 15.567,8 MM: el 40 % de su monto real. La misma columna había producido los 4.335,4 MM de F01, que son 4.478,8 |
| F15 | Reportar «el resultado del modelo de A» con una sola cifra | Los pasos 7, 11, 12 y 13 calculan la misma especificación de 141 variables y dan 0,842, 0,818, 0,830 y 0,835, porque cada uno usa distinto número de bolsas y su propia muestra de no etiquetados | Al cuadrar las cuatro salidas de la misma corrida | El paso 7b es el protocolo canónico; el valor p sale del paso 12 comparado contra su propio observado | Dispersión entre pasos 0,024, menor que el recorrido entre semillas 0,034 |
| F16 | Elegir la mejor de las cuatro especificaciones de A mirando los resultados | Con 15 conglomerados la elección es ruido: en doce particiones el ganador rotó 4-3-3-2 entre los cuatro conjuntos. Y elegir el mejor infla la cifra 0,108 sin comprar nada: el ganador da 0,817 donde se eligió y 0,709 en la otra mitad, mientras `A`, que no se elige, da 0,718 | Especificación por mitades repetida, cuaderno 11 pieza 3 | `A` es la especificación primaria declarada por origen; las otras tres son robustez, sin afirmar que alguna sea mejor | `A_anticipada` ganó tres particiones con 0,929, 0,914 y 0,723 y dio 0,411, 0,509 y 0,403 en la otra mitad. En los datos completos es nula: 0,526 |
| F17 | Citar «5.070.819 releases, 1,82 por proceso» como el tamaño de la descarga | La cifra no se podía reproducir. Un recorrido completo de los doce archivos, release por release, cuenta 2.785.355 releases y 2.785.355 procesos: exactamente uno por proceso. El portal publica un release compilado por `ocid`, no la secuencia de eventos | Al verificar el supuesto de que había varios releases por proceso para medir cambios de plazo o de monto (cuaderno 14) | La cifra se retira. Se declara como límite del dato que con un solo release por proceso no se puede observar si un plazo o un monto cambió, y que ningún archivo trae modificaciones de contrato, de adjudicación o de pliego ni datos de ejecución | 5.070.819 → 2.785.355 (1,82 → 1,00 releases por proceso) |
| F18 | Afirmar que el portal no publica procesos desiertos, cancelados ni suspendidos, porque no aparecen en los datos abiertos | Lo primero es cierto de los datos abiertos OCDS; lo segundo es falso: la ficha del portal transaccional publica el campo «Estado del Proceso» (*Desierta*, *Suspendido*, *Ejecución de Contrato*) con su razón, su justificación legal y su fecha | Revisión manual en el portal, el 9 de octubre de 2026, de procesos que en los datos no tienen adjudicación ni contrato | Se escribe: los datos abiertos OCDS no transportan el estado de desierto, cancelado ni suspendido, aunque el portal lo publica. La tasa de desiertos y la validación por desenlaces quedan como trabajo futuro (exigen raspar el portal) | Tres casos verificados: `LCS-EPCONST-002-2021` (105,1 MM, desierta), `LICS-EPP-2023-041` (34,6 MM, desierta), `LICO-GADMCA-2026-001` (20,2 MM, suspendido). En el OCDS, `tender.status` solo trae *complete* (2.598.496), *active* (137.891) o vacío (48.968), y los tres casos figuran como *active*, con solo planificación y licitación |

Las que siguen son las demostraciones reproducibles de las más instructivas.

In [ ]:
# ============ Entorno ============
import os, sys, json, time, io, contextlib
import numpy as np, pandas as pd, duckdb
os.chdir(r'C:\\SERCOP_FINAL')
R = '03_resultados'; OUT = f'{R}/c8_'; OUT9 = f'{R}/c9_'
con = duckdb.connect(); con.sql('SET preserve_insertion_order=false')
con.sql(f"CREATE OR REPLACE VIEW bm4 AS SELECT * FROM read_parquet('{R}/base_maestra_v4.parquet')")
def auc(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y).astype(int)
    m = ~np.isnan(x); x, y = x[m], y[m]
    if y.sum() == 0 or y.sum() == len(y): return np.nan
    r = pd.Series(x).rank().values; n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1*(n1+1)/2) / (n1*n0))
print('Listo.', con.sql('SELECT count(*) FROM bm4').fetchone()[0], 'registros en la base maestra')

## F01 · El RUC en cadena vacía

El filtro `proveedor_ruc IS NOT NULL` deja pasar la cadena vacía. Como la cadena vacía es un
valor válido, al agrupar por RUC esas 42.294 filas se convierten en **un solo proveedor**, que
además resulta el más grande del país. Entró así a la red bipartita.

La demostración calcula la concentración del gasto con y sin el filtro corregido.

In [ ]:
# --- F01 ---
q = lambda cond: con.sql(f"""
WITH p AS (SELECT proveedor_ruc, sum(monto_final) m FROM bm4 WHERE {cond} GROUP BY 1 HAVING m > 0),
     r AS (SELECT m, row_number() OVER (ORDER BY m DESC) pos, count(*) OVER () n, sum(m) OVER () tot FROM p)
SELECT count(*) proveedores, round(max(tot)/1e6,1) millones,
       round(sum(CASE WHEN pos <= n*0.01 THEN m ELSE 0 END)/max(tot)*100,1) top1
FROM r""").df()
a = q("proveedor_ruc IS NOT NULL");                      a.insert(0,'filtro','IS NOT NULL (el malo)')
b = q("TRIM(COALESCE(proveedor_ruc,'')) <> ''");         b.insert(0,'filtro','TRIM <> char vacio')
print(pd.concat([a,b]).to_string(index=False))
print()
print(con.sql("""SELECT count(*) filas, round(sum(monto_final)/1e6,1) millones
  FROM bm4 WHERE proveedor_ruc IS NOT NULL AND TRIM(proveedor_ruc) = ''""").df().to_string(index=False))
print('\nEsas filas se agrupaban como un unico proveedor. Era el mas grande del pais.')

### Verificación del monto (corrección de F01)

La cifra que figuraba en el registro, 4.335,4 millones, salía de `monto_tender`. Esta celda suma
las tres columnas de monto sobre los mismos 46.370 registros sin RUC y confirma cuál es la correcta.

In [ ]:
# verificacion del monto de los registros sin RUC (F01)
print(con.sql("""
SELECT CASE WHEN proveedor_ruc IS NULL THEN 'nulo' ELSE 'cadena vacia' END AS tipo,
       count(*) AS filas,
       round(sum(monto_final)/1e6, 1) AS final_MM,
       round(sum(monto_tender)/1e6, 1) AS tender_MM,
       round(sum(monto_award)/1e6, 1) AS award_MM,
       count(monto_final) AS valores_final
FROM bm4
WHERE proveedor_ruc IS NULL OR TRIM(proveedor_ruc) = ''
GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

## F04 · El AUC calculado sobre la población equivocada

Para ser positivo del objetivo B hay que tener grupo societario **y** aparecer en la tabla de
oferentes. Calcular el AUC sobre todo el panel mete en la comparación a más de 215.000 filas que
nunca pudieron ser positivas, y eso infla cualquier variable que se correlacione con pertenecer
al universo.

In [ ]:
# --- F04 ---
P = pd.read_parquet(f'{OUT}panel.parquet')
uB = (P.tiene_grupo & P.aparece_oferentes).values.astype(bool)
y  = P.y_intragrupo.values.astype(int)
print(f'panel completo: {len(P):,} filas | universo de B: {uB.sum():,} filas')
filas = []
for c in ['sup_anonima','sup_responsabilidad_limi','n_parties','proveedor_unico']:
    if c in P.columns:
        filas.append((c, round(auc(P[c].values, y), 3),
                         round(auc(P[c].values[uB], y[uB]), 3)))
print(pd.DataFrame(filas, columns=['variable','auc_todo_el_panel','auc_universo_B']).to_string(index=False))
print('\nLa primera columna es la trampa: no es que la variable prediga, es que marca quien pertenece.')

## F06 · La codificación mal declarada no da error, da menos datos

El lector de CSV admite ignorar errores. Con la codificación equivocada no falla: descarta en
silencio las filas que no puede interpretar. Aquí se lee el mismo archivo de las dos maneras.

In [ ]:
# --- F06 ---
import glob
cand = sorted(glob.glob('05_documentos/SRI/SRI_RUC_*.csv'), key=os.path.getsize)
if not cand:
    print('No hay archivos en 05_documentos/SRI: esta demostracion necesita los CSV del catastro.')
    arch = None
else:
    arch = cand[0]
    print('Archivo de prueba (el mas chico):', arch, f'{os.path.getsize(arch)/1e6:.1f} MB')
filas = []
for enc in (['utf-8','latin-1'] if arch else []):
    try:
        n = con.sql(f"""SELECT count(*) FROM read_csv('{arch.replace(chr(92),'/')}', delim='|',
              header=true, encoding='{enc}', all_varchar=true, ignore_errors=true)""").fetchone()[0]
    except Exception as e:
        n = f'rechazado: {str(e)[:45]}'
        try: con.rollback()          # si no, la transaccion queda abortada y rompe las celdas siguientes
        except Exception: pass
    filas.append((enc, n))
d = pd.DataFrame(filas, columns=['encoding','filas_leidas'])
if len(d): print(d.to_string(index=False))
print('''
Sobre UN archivo, el lector rechaza la codificacion equivocada: eso es lo deseable.
Lo que nos paso fue distinto: se leyeron los 24 archivos de una sola vez, con un comodin y
con la opcion de ignorar errores activada. Ahi no hubo mensaje, hubo menos filas:

  lectura declarando latin-1 ....... 1.759.893 filas
  lectura declarando utf-8 ......... 8.458.107 filas

El 79 % se perdio en silencio, y la unica senal visible era que no habia ni un solo nombre
con tilde ni con n con virgulilla en 1,7 millones de registros.''')

## F07 · El nulo mal calibrado

Validar dejando un conglomerado fuera desplaza el estadístico por sí solo: el positivo retenido
lo puntúa un modelo que no lo vio. Por eso el nulo correcto no está en 0,50. Comparar contra 0,50
hacía que el control negativo pareciera fallar.

Aquí se superponen los cinco placebos del control negativo con el intervalo del nulo de
permutación.

In [ ]:
# --- F07 ---
try:
    rob = pd.read_csv(f'{OUT}robustez.csv')
    pm  = pd.read_csv(f'{OUT}permutacion.csv')
    pl  = rob[rob.prueba.str.contains('placebo|control|negativ', case=False, na=False)] if 'prueba' in rob.columns else rob
    nA  = pm[(pm.caso.str.startswith('A fantasma · completo')) & (pm.metodo == 'modelo')].iloc[0]
    print(f"Nulo de permutacion del objetivo A: mediana {nA['nulo_mediana']:.3f} "
          f"| intervalo 95 % [{nA['nulo_p2.5']:.3f}, {nA['nulo_p97.5']:.3f}]")
    print(f"Observado: {nA['observado']:.3f}  ->  p = {nA['p_una_cola']:.2f}")
    print('\nPlacebos del control negativo (deberian caer DENTRO del intervalo):')
    print(pl.to_string(index=False))
except FileNotFoundError as e:
    print('Falta un archivo:', e, '\nCorre los pasos 11 y 12 del cuaderno 8 primero.')

## F08 · `monto_tender`, el atajo involuntario

`monto_tender` está nulo en el 100 % de las subastas inversas y presente en el 100 % de las otras
cuatro clases de procedimiento. Su patrón de nulos **es** el tipo de procedimiento. El modelo
podía usarlo para aprender «este proveedor no usa subastas inversas» en lugar de conducta.

Lo notable: al quitarlo, el resultado en los universos restringidos **mejoró**.

La segunda parte de la celda es **F14**, que es la misma falla vista desde otro lado: mientras
`monto_tender` seguía en la base, la tabla de montos por clase de procedimiento del documento de
trazabilidad se calculó con él. Sumar una columna que está nula en el 84,7 % del universo
competitivo no produce un total más bajo: produce el total de otro conjunto de procesos.

In [ ]:
# --- F08 ---
try: con.rollback()                    # por si una celda anterior dejo la transaccion abortada
except Exception: pass
cols_bm4 = {r[0] for r in con.sql('DESCRIBE bm4').fetchall()}
try:
    assert 'monto_tender' in cols_bm4, 'la base no tiene monto_tender'
    pr = pd.read_parquet(f'{OUT9}procedimiento.parquet', columns=['ocid','clase'])
    mt = con.sql('SELECT ocid, monto_tender, monto_final FROM bm4').df()
    j  = pr.merge(mt, on='ocid', how='inner')
    t  = j.groupby('clase').agg(procesos=('ocid','size'),
                                pct_nulo=('monto_tender', lambda s: round(s.isna().mean()*100,1)))
    print(t.to_string())
    print('\nCero o cien: el nulo no esta repartido, identifica la clase.')

    # --- F14: la consecuencia en las cifras ya publicadas ---
    g = j.groupby('clase').agg(tender_MM=('monto_tender', lambda s: round(s.sum()/1e6,1)),
                               final_MM =('monto_final',  lambda s: round(s.sum()/1e6,1)),
                               procesos_con_tender=('monto_tender', lambda s: int(s.notna().sum())),
                               procesos=('ocid','size'))
    g['pct_procesos_sumados'] = (g.procesos_con_tender / g.procesos * 100).round(1)
    print('\nF14 - la misma tabla de montos, con cada columna:')
    print(g.to_string())
    print(f'\nTotal con monto_tender: {j.monto_tender.sum()/1e6:,.1f} MM | '
          f'con monto_final: {j.monto_final.sum()/1e6:,.1f} MM  (suma exacta, no de las filas redondeadas)')
    print('Sumar monto_tender no da un total menor por redondeo: da el total de UNA PARTE')
    print('de los procesos, y esa parte la elige el tipo de procedimiento.')
except (FileNotFoundError, AssertionError) as e:
    print('No se pudo mostrar la tabla:', e)
print('''
Y el efecto de quitarlo, medido con el nulo de permutacion:

  caso                              con monto_tender   sin monto_tender
  A universo de oferentes           0,760  (p 0,06)    0,797  (p 0,04)
  A sin ausencia informativa        0,738  (p 0,08)    0,804  (p 0,04)

Quitar la variable que confundia con el procedimiento hizo que el modelo generalizara MEJOR.
Es la demostracion de por que la exclusion se decide por origen y no por desempeno.''')

## F09 · El azar global rompe la reproducibilidad

La función de evaluación tomaba la muestra de comparación y la partición de pliegues del
generador de azar **global**. Eso hace que el resultado dependa de cuántas celdas consumieron
azar antes, es decir del orden en que se corrió el cuaderno.

Esta demostración no necesita los datos: reproduce el mecanismo en diez líneas.

In [ ]:
# --- F09 ---
def con_azar_global(rng):               # como estaba
    return rng.integers(0, 1000, 3).tolist()
def con_semilla_explicita(semilla):     # como quedo
    return np.random.default_rng(semilla).integers(0, 1000, 3).tolist()

print('Con el generador global, segun cuantas celdas corrieron antes:')
for celdas_antes in [0, 1, 5]:
    g = np.random.default_rng(20261001)
    for _ in range(celdas_antes): g.random(10)          # otra celda que consume azar
    print(f'  {celdas_antes} celdas antes -> {con_azar_global(g)}')
print('\nCon semilla explicita, siempre lo mismo:')
for celdas_antes in [0, 1, 5]:
    print(f'  {celdas_antes} celdas antes -> {con_semilla_explicita(20261001)}')
print('''
En el cuaderno esto se veia asi: los casos "anticipada", con EXACTAMENTE las mismas 46
variables y los mismos datos, dieron 0,673 en una corrida y 0,422 en la siguiente. Los
indices, que no se entrenan, dieron casi identico (0,870 y 0,868): prueba de que los datos
eran los mismos y lo que se movia era el modelo.''')

## F10 y F11 · El catálogo electrónico lo explicaba todo

El 83,5 % de los procesos son compras por catálogo electrónico, donde no hay proceso de ofertas.
Eso diluía la tasa de oferente único y explicaba, por sí solo, el 100 % de los nulos de
`n_oferentes` que llevábamos semanas tratando como problema de calidad.

In [ ]:
# --- F10 y F11 ---
try: con.rollback()
except Exception: pass
try:
    pr = pd.read_parquet(f'{OUT9}procedimiento.parquet', columns=['ocid','clase','es_catalogo'])
    bb = con.sql('SELECT ocid, n_oferentes, proveedor_unico, monto_final FROM bm4').df()
    j  = pr.merge(bb, on='ocid', how='inner')
    # proveedor_unico es booleano anulable: pasarlo a float para poder promediar
    j['proveedor_unico'] = pd.to_numeric(j.proveedor_unico, errors='coerce').astype(float)
    t  = j.groupby('clase').agg(procesos=('ocid','size'),
            nulos_n_oferentes=('n_oferentes', lambda s: int(s.isna().sum())),
            pct_oferente_unico=('proveedor_unico', lambda s: round(s.mean()*100,1)),
            millones=('monto_final', lambda s: round(s.sum()/1e6,1)))
    print(t.to_string())
    tot = int(j.n_oferentes.isna().sum()); cat = int(j.loc[j.es_catalogo,'n_oferentes'].isna().sum())
    print(f'\nNulos de n_oferentes: {tot:,}' +
          (f' | del catalogo: {cat:,} ({cat/tot*100:.1f} %)' if tot else ' (ninguno)'))
    print(f'Oferente unico, toda la base: {j.proveedor_unico.mean()*100:.1f} %')
    print(f'Oferente unico, sin catalogo: {j.loc[~j.es_catalogo,"proveedor_unico"].mean()*100:.1f} %')
except FileNotFoundError:
    print('Falta c9_procedimiento.parquet: corre el paso 5b del cuaderno 9.')

## F13 · El mismo dato, tres cifras distintas

Esta falla la encontró este cuaderno mientras lo escribíamos, y es la más incómoda de todas,
porque la cifra equivocada ya estaba reportada.

En el cuaderno 9 calculamos la tasa de oferente único con SQL:

```sql
avg(CASE WHEN proveedor_unico THEN 1.0 ELSE 0.0 END)
```

Esa expresión tiene un problema que no se ve: cuando `proveedor_unico` es **nulo**, no cae en el
`THEN` ni queda fuera del promedio — cae en el `ELSE 0.0`. Es decir, el SQL afirma «no hubo
oferente único» en los 2.285.455 procesos de catálogo electrónico, donde en realidad **no hay
proceso de ofertas y por tanto no hay nada que afirmar**. El resultado fue 7,7 %, y lo
interpretamos como «la tasa real, diluida por el catálogo».

No era una tasa diluida. Era la tasa verdadera repartida sobre un denominador que no le
correspondía. La celda de abajo lo demuestra de dos maneras: calculando lo mismo en pandas, que
sí omite los nulos, y comprobando la identidad aritmética — 47,0 % × 450.913 / 2.736.387 da
exactamente el 7,7 % que habíamos reportado.

De aquí sale la regla que aplicamos en el resto de la tesis: **una tasa se reporta únicamente
sobre el universo donde la variable existe, y ese universo se declara junto a la cifra.** Para el
oferente único ese universo es el procedimiento competitivo abierto, donde la tasa es 22,6 %.


In [ ]:
# --- F13 ---
try: con.rollback()
except Exception: pass
try:
    pr = pd.read_parquet(f'{OUT9}procedimiento.parquet', columns=['ocid','clase','es_catalogo'])
    bb = con.sql('SELECT ocid, proveedor_unico FROM bm4').df()
    j  = pr.merge(bb, on='ocid', how='inner')
    j['pu'] = pd.to_numeric(j.proveedor_unico, errors='coerce').astype(float)

    # 1) Exactamente la expresion del cuaderno 9: el nulo cae en el ELSE 0
    sql = con.sql('SELECT avg(CASE WHEN proveedor_unico THEN 1.0 ELSE 0.0 END)*100 FROM bm4').fetchone()[0]

    n_tot = len(j); n_nul = int(j.pu.isna().sum()); n_obs = n_tot - n_nul
    obs  = j.pu.mean() * 100                                   # pandas omite los nulos
    sin  = j.loc[~j.es_catalogo, 'pu'].mean() * 100
    comp = j.loc[j.clase.str.contains('abierto'), 'pu'].mean() * 100

    print(f'Procesos: {n_tot:,} | con valor: {n_obs:,} | nulos: {n_nul:,} ({n_nul/n_tot*100:.1f} %)')
    print('\nLa misma pregunta, cuatro respuestas:')
    print(f'  1. SQL, el nulo contado como "no" : {sql:5.1f} %   <- la cifra que reportamos')
    print(f'  2. pandas, el nulo omitido        : {obs:5.1f} %')
    print(f'  3. Solo donde la variable existe  : {sin:5.1f} %')
    print(f'  4. Universo competitivo abierto   : {comp:5.1f} %   <- la que se reporta')
    print(f'\nIdentidad aritmetica: {obs:.1f} % x {n_obs:,} / {n_tot:,} = {obs*n_obs/n_tot:.2f} %')
    print(f'Esa es la cifra 1. No era una tasa diluida: era la tasa real repartida sobre')
    print(f'los {n_nul:,} procesos en los que la variable no existe.')
    print('\nY el aviso que el SQL no da: cuantos valores entraron en cada promedio')
    msk = {'toda la base': pd.Series(True, index=j.index),
           'sin catalogo': ~j.es_catalogo,
           'competitivo' : j.clase.str.contains('abierto')}
    for nom, m in msk.items():
        s = j.loc[m, 'pu']
        print(f'  {nom:14s}: {len(s):>9,} filas | {int(s.notna().sum()):>9,} con valor')
except FileNotFoundError:
    print('Falta c9_procedimiento.parquet: corre el paso 5b del cuaderno 9.')

## F15 · Cuatro cifras para el mismo resultado

Al correr los pasos 7, 11, 12 y 13 con el conjunto corregido de 141 variables, los cuatro
reportaron el objetivo A completo (la misma especificación, los mismos datos, la misma semilla
global) con cuatro valores distintos:

| Paso | Qué hace ahí | Mediana del percentil |
| --- | --- | --- |
| 11 | Control negativo y estabilidad entre semillas | 0,818 |
| 12 | Nulo por permutación de etiquetas | 0,830 |
| 13.1 | Comportamiento frente a cobertura | 0,835 |
| **7b** | **Evaluación reproducible con 5 semillas** | **0,842** · recorrido [0,825 – 0,859] |

No es una contradicción. Cada paso usa un número distinto de bolsas de PU bagging y su propia
muestra de no etiquetados, de modo que estima la misma cantidad con distinta cantidad de
promediado. La dispersión entre pasos, 0,024, es **menor** que el recorrido entre semillas de un
solo paso, 0,034: todo cae dentro del ruido del procedimiento.

Lo que sí es un problema es reportarlo sin decirlo. De aquí salen dos reglas:

1. **La cifra que va en la tesis es la del paso 7b**, con su recorrido entre semillas al lado.
   Es el único paso con semillas explícitas y repetición declarada.
2. **El valor p sale del paso 12 comparado contra el observado del paso 12**, no contra la media
   del paso 7b. Un valor p solo tiene sentido si el observado y el nulo vienen del mismo
   procedimiento; mezclarlos es comparar una medición contra la distribución de otra.

La única especificación donde la diferencia entre protocolos se sale del recorrido es *A universo
de oferentes*: el paso 7b da 0,754 con recorrido [0,736 – 0,768] y el paso 12 observa 0,797. Ahí
hay que elegir explícitamente y declararlo, porque los dos números no son intercambiables.

**Nota sobre el valor p.** El paso 12 usó 49 permutaciones y dio p = 0,02. El cuaderno 13 lo repitió con 199 permutaciones: para A completo el observado es 0,830 contra un nulo de mediana 0,441 y p = 0,005, el mínimo alcanzable. Se cita solo el de 199 y se aclara que el nulo no está centrado en 0,5.

In [ ]:
# --- F15 ---
def mostrar(nombre, patron=None):
    ruta = f'{OUT}{nombre}'
    if not os.path.exists(ruta):
        print(f'\n{nombre}: falta, corre el paso correspondiente'); return
    d = pd.read_csv(ruta)
    print(f'\n--- {nombre} ---')
    if patron:
        c = [x for x in d.columns if x.lower() in ('caso', 'objetivo')]
        if c:
            d = d[d[c[0]].astype(str).str.contains(patron, case=False, na=False, regex=True)]
    print(d.to_string(index=False) if len(d) else '  (ninguna fila coincide)')

print('La MISMA especificacion -- objetivo A, conjunto completo de 141 variables --')
print('tal como la reporta cada paso del cuaderno 8:\n')
print('  paso 11 (control negativo) ....... 0,818')
print('  paso 12 (permutacion) ............ 0,830')
print('  paso 13.1 (cobertura) ............ 0,835')
print('  paso 7b (canonico, 5 semillas) ... 0,842   recorrido [0,825 - 0,859]')
print('\n  dispersion entre pasos: 0,024 | recorrido entre semillas: 0,034')

mostrar('resultados_v2.csv', r'A .*completo')
mostrar('permutacion.csv',   r'A .*completo')
mostrar('permutacion_199.csv')   # version final: 199 permutaciones (cuaderno 13)
mostrar('robustez.csv',      r'^A$')
mostrar('cobertura_universo.csv')

print('''
Las cuatro cifras no se contradicen: son la misma cantidad estimada con distinto numero
de bolsas de PU bagging y distinta muestra de no etiquetados. La dispersion entre pasos
(0,024) es MENOR que el recorrido entre semillas de un solo paso (0,034), asi que todo
cae dentro del ruido del procedimiento.

Pero la tesis reporta UNA cifra, y tiene que decir cual protocolo la produjo:

  valor reportado -> paso 7b: 25 bolsas, 5 semillas, recorrido declarado
  valor p         -> paso 12, comparado contra SU PROPIO observado

Lo segundo no es un detalle. Un valor p solo es valido si el observado y el nulo salen
del mismo procedimiento. Tomar el p del paso 12 y pegarlo junto a la media del paso 7b
seria comparar una medicion contra la distribucion de otra.''')

## F16 · Elegir la mejor especificación mirando el resultado

Para el objetivo A hay cuatro conjuntos de variables posibles (`A`, `A_sin_ausencia`,
`A_anticipada` y `A_estricta`). Con solo 15 conglomerados positivos, la tentación es quedarse con
el que mejor sale. La prueba (cuaderno 11, pieza 3) parte los conglomerados positivos en dos
mitades, elige el conjunto mirando una mitad y lo mide en la otra, y se repite 12 veces con
particiones distintas. El archivo `c8_mitades.csv` guarda las 12.

Dos hallazgos:

- **El ganador rota casi uniformemente** (4, 3, 3 y 2 de las doce particiones). La elección entre
  las cuatro especificaciones es ruido.
- **El optimismo por selección es 0,108.** El ganador da 0,817 en la mitad donde se eligió y
  0,709 en la que no se usó. El conjunto `A`, que no se elige por desempeño, da 0,718 en esa
  mitad no usada, es decir, un poco por encima del «ganador».

`A_anticipada` es la demostración más limpia: ganó tres particiones con 0,929, 0,914 y 0,723 y
en la otra mitad dio 0,411, 0,509 y 0,403, siendo el mismo conjunto que en los datos completos
es nulo (0,526). Una especificación nula puede verse excelente en una mitad afortunada.

**Decisión.** `A`, el conjunto completo definido por origen, es la especificación primaria
declarada de antemano. Las otras tres se reportan como robustez, sin afirmar que alguna sea
mejor. Así no hay selección y el 0,108 no se aplica al número principal.

In [ ]:
# --- F16 ---
ruta = f'{OUT}mitades.csv'
if not os.path.exists(ruta):
    print('Falta c8_mitades.csv: corre la pieza 3 del cuaderno 11.')
else:
    m = pd.read_csv(ruta)
    print(f'Particiones: {len(m)}  |  conglomerados por mitad: {int(m.n_m1.iloc[0])} y {int(m.n_m2.iloc[0])}\n')
    print('Veces que gana cada conjunto en la mitad donde se elige:')
    print(m.ganador_en_m1.value_counts().to_string())

    en_m1 = m.valor_m1.median()          # el ganador, en la mitad donde se eligio
    en_m2 = m.ganador_en_m2.median()     # el ganador, en la mitad que no se uso
    a_m2  = m.A_en_m2.median()           # el conjunto A (no se elige), en la mitad que no se uso
    print(f'\nGanador, mitad donde se eligio ..... mediana {en_m1:.3f}  [{m.valor_m1.min():.3f} - {m.valor_m1.max():.3f}]')
    print(f'Ganador, mitad que no se uso ....... mediana {en_m2:.3f}  [{m.ganador_en_m2.min():.3f} - {m.ganador_en_m2.max():.3f}]')
    print(f'Conjunto A, mitad que no se uso .... mediana {a_m2:.3f}  [{m.A_en_m2.min():.3f} - {m.A_en_m2.max():.3f}]')
    print(f'\nOptimismo por seleccion (elegido - no usado): {en_m1 - en_m2:.3f}')

    print('\nEl caso de A_anticipada (una especificacion nula en los datos completos):')
    x = m[m.ganador_en_m1 == 'A_anticipada'][['particion', 'valor_m1', 'ganador_en_m2']]
    x.columns = ['particion', 'valor_donde_se_eligio', 'valor_en_la_otra_mitad']
    print(x.to_string(index=False))
    print('\nElegir el mejor de cuatro no compra nada fuera de muestra: solo infla el numero dentro de ella.')

## F17 · Una cifra de procedencia que no se pudo reproducir

El documento de trazabilidad decía que se habían descargado **5.070.819 releases, 1,82 por
proceso**. Esa cifra salió del cuaderno 9 y no se pudo identificar qué contaba. Un recorrido
completo, release por release, de los doce archivos (celda de verificación del cuaderno 14) cuenta
**2.785.355 releases y 2.785.355 procesos: exactamente uno por proceso**. La cifra anterior queda
retirada.

Lo importante no es el número sino lo que implica. Los archivos del portal traen un release
compilado por `ocid`, no la secuencia de eventos de cada proceso. Por eso:

- **No se puede observar si un plazo o un monto cambió en el tiempo.**
- **Ningún archivo publica modificaciones de contrato, de adjudicación o de pliego, ni datos de
  ejecución** (cero en los doce años).

Eso no es un defecto del análisis sino un límite del dato, y se declara como tal.

In [ ]:
# --- F17 ---
ruta = f'{R}/c14_verificacion_releases.csv'
if not os.path.exists(ruta):
    print('Falta c14_verificacion_releases.csv: corre la celda de verificacion del cuaderno 14.')
else:
    v = pd.read_csv(ruta)
    print(v[['archivo', 'releases', 'procesos', 'releases_por_proceso']].to_string(index=False))
    tot_r, tot_p = int(v.releases.sum()), int(v.procesos.sum())
    print(f'\nTotal: {tot_r:,} releases | {tot_p:,} procesos | {tot_r / tot_p:.2f} releases por proceso')
    print('Archivos con mas de un release por proceso:', int((v.releases_por_proceso > 1).sum()))
    print(f'La cifra que figuraba antes era 5.070.819 ({5070819 / tot_p:.2f} por proceso): queda retirada.')
    print('(Que ningun proceso se repita entre archivos lo verifica la consolidacion por ocid del cuaderno 9.)')
    ausentes = ['con_implementacion', 'con_transacciones', 'con_hitos',
                'con_modif_contrato', 'con_modif_adjudicacion', 'con_modif_pliego']
    print('\nProcesos que traen cada elemento, suma de los doce archivos:')
    print(v[ausentes].sum().astype(int).to_string())
    print('\nCon un solo release por proceso y estos ceros, el dato abierto no permite medir cambios')
    print('de plazo, de monto ni de condiciones. Se declara como limite, no se rellena.')

## F18 · Lo que el dato abierto no trae no es lo que el portal no publica

Hasta el 9 de octubre los documentos decían que «ningún proceso aparece desierto o cancelado» y que
«el portal no publica» esos estados. Lo primero es cierto de los datos abiertos OCDS; **lo segundo
es falso**. Al revisar a mano en el portal transaccional procesos que en los datos no tienen
adjudicación ni contrato, la ficha mostró el campo **«Estado del Proceso»** con valores *Desierta*,
*Suspendido* y *Ejecución de Contrato*, con su razón, su justificación legal y su fecha:

| Proceso | Monto | Estado en el portal |
|---|---|---|
| `LCS-EPCONST-002-2021` | 105,1 MM | Desierta el 8-dic-2021 (art. 33, lit. b, de la LOSNCP; ofertas inhabilitadas) |
| `LICS-EPP-2023-041` | 34,6 MM | Desierta el 28-mar-2024 («inconvenientes para los intereses nacionales o institucionales») |
| `LICO-GADMCA-2026-001` | 20,2 MM | Suspendido |

La redacción correcta es: **los datos abiertos OCDS no transportan el estado de desierto, cancelado
ni suspendido, aunque el portal transaccional lo publica en la ficha del proceso.** Es una brecha
entre dos sistemas de publicación del propio SERCOP.

Consecuencia: la tasa de desiertos de la entidad y la validación por desenlaces no son inviables
en absoluto, sino **inviables con los datos abiertos**; exigirían raspar el portal transaccional, lo
que queda fuera del alcance y se declara como trabajo futuro. Los 3.234 procesos sin adjudicación
ni contrato iniciados más de 180 días antes del corte (1.541,1 millones; registro de cifras,
sección 14) son candidatos a desierto o cancelado que el dato abierto no permite clasificar.

La corrida del 25 de octubre lo confirma: en los 2.785.355 procesos, `tender.status` solo vale
*complete* (2.598.496), *active* (137.891) o está vacío (48.968, los solo planeación y sin monto);
ningún proceso trae un estado de cancelado, desierto o suspendido. Los tres casos verificados
figuran en el dato abierto como **activos**, con solo las etapas de planificación y licitación, aunque
dos de ellos se declararon desiertos en 2021 y 2024.

La lección es de redacción: **que algo no esté en el archivo no autoriza a decir qué publica o no
publica la fuente.** La celda muestra los estados que sí trae el dato abierto y cómo aparecen ahí
los tres casos verificados.

In [ ]:
# --- F18 ---
# Qué estados del proceso trae el dato abierto, y cómo aparecen ahí los tres casos que el portal
# muestra como desiertos o suspendidos. Solo lee planA_universo; no imprime RUC.
U = 'plan_A/planA_universo.parquet'
cols = [r[0] for r in con.sql(f"DESCRIBE SELECT * FROM read_parquet('{U}')").fetchall()]
ajenas = ('proveedor', 'entidad', 'sri', 'sup', 'y_')            # estados del SRI o de la Superintendencia
est = [c for c in cols if ('status' in c.lower() or 'estado' in c.lower()) and not c.lower().startswith(ajenas)]
print('Columnas de estado del proceso en los datos abiertos:', est if est else 'ninguna')
n_u = con.sql(f"SELECT count(*) FROM read_parquet('{U}')").fetchone()[0]
for c in est:
    print(f'\n{c} (universo: {n_u:,} procesos)')
    print(con.sql(f"""SELECT coalesce(CAST("{c}" AS VARCHAR), '(vacío)') AS valor, count(*) AS procesos
                      FROM read_parquet('{U}') GROUP BY 1 ORDER BY 2 DESC LIMIT 15""").df().to_string(index=False))
if est:
    buscar = " OR ".join(f"""lower(CAST("{c}" AS VARCHAR)) SIMILAR TO '.*(cancel|unsuccess|withdrawn|desiert|suspend|anulad).*'"""
                         for c in est)
    n = con.sql(f"SELECT count(*) FROM read_parquet('{U}') WHERE {buscar}").fetchone()[0]
    print(f'\nProcesos con un estado de cancelado, desierto, suspendido o retirado en esas columnas: {n:,}')

casos = {'LCS-EPCONST-002-2021': 'Desierta el 8-dic-2021',
         'LICS-EPP-2023-041': 'Desierta el 28-mar-2024',
         'LICO-GADMCA-2026-001': 'Suspendido'}
mostrar = [c for c in ['ocid'] + est + ['etapas', 'monto_final'] if c in cols]
print('\nLos tres casos verificados a mano en el portal el 9 de octubre, tal como están en el dato abierto:')
for cod, portal in casos.items():
    t = con.sql(f"""SELECT {', '.join(f'"{c}"' for c in mostrar)} FROM read_parquet('{U}')
                    WHERE ocid LIKE '%{cod}%'""").df()
    if 'monto_final' in t:
        t['monto_final'] = (t['monto_final'] / 1e6).round(1).astype(str) + ' MM'
    print(f'\n  {cod} | en el portal: {portal}')
    print('  ' + (t.astype(str).to_string(index=False).replace('\n', '\n  ') if len(t) else 'no aparece en el universo'))
print('\nLo que el archivo no trae no autoriza a decir que la fuente no lo publica: el estado existe en')
print('la ficha del portal transaccional, no en los datos abiertos. Se declara como límite de los datos abiertos.')

## Cierre · Qué se lleva el lector

1. **Ningún error se encontró mirando el resultado final.** Todos salieron de auditar el
   procedimiento: el filtro, la población de comparación, la codificación, el patrón de nulos,
   el generador de azar.
2. **Dos correcciones mejoraron el resultado en lugar de empeorarlo** (F08 y F01). Eso es lo que
   se espera cuando se quita un confundidor y no una señal.
3. **Una corrección cambió una conclusión** (F07): el control negativo no fallaba, fallaba la
   referencia contra la que lo comparábamos.
4. **Dos «problemas de calidad» no lo eran** (F10 y F11): eran propiedades del procedimiento de
   contratación que no habíamos entendido.
5. **El propio registro tenía un error.** La entrada F04 citaba de memoria una cifra que no
   coincidía con los datos; la demostración la corrigió. Un registro de fallas sirve solo si sus
   números se vuelven a medir, no si se copian.
6. **Y una cifra que habíamos reportado resultó no serlo** (F13): el 7,7 % de oferente único
   sobre toda la base no era una tasa diluida, era el efecto de contar como «no» dos millones de
   valores que no existen. Lo encontró este mismo cuaderno, al recalcular en pandas lo que el SQL
   había calculado. **Cuando dos herramientas dan números distintos sobre el mismo dato, la
   diferencia casi siempre está en el nulo.**
7. **Y la misma variable nula arrastró la tabla de montos** (F14): el universo competitivo
   aparecía con 22.974,4 millones cuando tiene 38.542,2, porque la suma de `monto_tender` solo
   recorre los procesos donde esa columna existe. F13 y F14 son el
   mismo error — **promediar o sumar sin declarar cuántos valores entraron** — y es el que más
   veces nos engañó. La tercera aparición fue la cifra de F01: los 4.335,4 millones de los
   registros sin RUC también salían de `monto_tender`; con `monto_final` son 4.478,8.
8. **Una regla que faltaba, no un error** (F15): cuatro pasos del cuaderno 8 calculan la misma
   especificación y dan cuatro cifras, todas dentro del ruido del procedimiento. **La tesis
   reporta una y dice qué protocolo la produjo.** El valor p se compara contra el observado de
   su propio procedimiento, nunca contra el de otro.
9. **Una entrada que no se encontró mirando los datos, sino el procedimiento de elegir** (F16):
   una especificación nula puede verse excelente en una mitad afortunada. `A_anticipada` llegó a
   0,929 en una mitad y a 0,411 en la otra, siendo el mismo conjunto que en los datos completos
   da 0,526. Es la razón por la que la tesis reporta la especificación declarada por origen y no
   la que mejor sale.
10. **Una cifra de procedencia que no se pudo reproducir** (F17): los 5.070.819 releases
    resultaron ser 2.785.355, uno por proceso. Lo que cambia no es el número sino lo que el dato
    permite: sin la secuencia de eventos no se pueden medir cambios de plazo ni de monto, y eso
    se declara como límite.
11. **Una afirmación sobre la fuente que el dato no sostenía** (F18): que un estado no aparezca
    en los datos abiertos no quiere decir que el SERCOP no lo publique. Tres procesos desiertos o
    suspendidos, verificados a mano en el portal, aparecen en el OCDS sin ese estado. Se declara
    como límite de los datos abiertos, no de la fuente.

> Nota: los placebos de la demostración F07 vienen de la corrida anterior a retirar
> `monto_tender`. Caen igual dentro del nuevo intervalo, pero conviene volver a correr el paso 11
> para que todo el documento use el mismo conjunto de variables.

El resultado que se reporta en la tesis es el que sobrevivió a las dieciocho entradas de este
registro. La semilla global de todos los procedimientos aleatorios es 20261001, y el manifiesto
`c9_manifiesto_archivos.csv` permite verificar que ningún archivo cambió desde entonces.